# 0. Data source
Set `USE_RAW` below. With `USE_RAW = False` (default), all Part 1 cells are skipped and the analysis uses the shipped consolidated `*_full.csv`.

In [ ]:
# Data source switch
#   USE_RAW = True : run Part 1 -- merge the raw per-run simulation outputs under
#                    2_prelimit_simulation/5_RS_saved_results_full/ (full local results, NOT shipped)
#                    and write the consolidated *_full.csv to cpp_results_formal_run/...;
#                    Part 2 then reads them from there.
#   USE_RAW = False: skip Part 1; Part 2 reads the shipped consolidated *_full.csv
#                    from 2_prelimit_simulation/3_RS_saved_results/.
USE_RAW = False

# 1. Merge Data

In [ ]:
if USE_RAW:
    import pandas as pd
    from pathlib import Path
    import re
    import os

    # search only under this directory
    root = Path(
        "../../2_prelimit_simulation/5_RS_saved_results_full/1_RS_proposed_policy_extra_5000p"
    )
    model_name = "RS"
    eps_band = "Out"
    results_folder_name = "cpp_results_formal_run/1_RS_proposed_policy_extra_5000p"
    figures_folder_name = "cpp_results_formal_run/1_RS_proposed_policy_extra_5000p"

    if not os.path.exists(results_folder_name):
        os.makedirs(results_folder_name)
    if not os.path.exists(figures_folder_name):
        os.makedirs(figures_folder_name)

In [ ]:
if USE_RAW:
    # the rhoadd section is optional
    pattern = re.compile(
        r"prelimit_simulation_results/proposed_policy_results/abs_"
        r"rho(?P<rho>[^/_]+)"
        r"(?:_rhoadd(?P<rho_add>[^/_]+))?"
        r"_safety(?P<safety>[^_]+)"
        r"_epsilon(?P<epsilon>[^/]+)"
        r"/RS_final_results\.csv$"
    )

    all_dfs = []

    for file_path in root.rglob("RS_final_results.csv"):

        path_str = str(file_path)

        match = pattern.search(path_str)

        # keep only those matching your directory structure
        if match:
            # df = pd.read_csv(file_path, header=None)
            df = pd.read_csv(file_path)

            # print(df.shape)
            # print(df.columns)
            # print(df.head())

            epsilon_str = match.group("epsilon")
            df["epsilon"] = float(epsilon_str)
            # df["rho"] = float(match.group("rho"))
            # df["lambda"] = float(match.group("lambda"))
            df["safety"] = float(match.group("safety").replace("n", "-"))

            all_dfs.append(df)

    # merge
    big_df = pd.concat(all_dfs, ignore_index=True)



    print("Number of files read:", len(all_dfs))
    print("Total rows:", len(big_df))
    print(big_df.columns)

    if model_name == "RS":
    
        big_df.columns = [
            "lambda","beta_tilde_1","beta_tilde_2","rho",
            'cost', 'std',
           'delay1', 'delay2', 'delay1_std', 'delay2_std', 'epsilon', 'safety'
        ]
    
    # elif model_name == "ECC3":
    
    #     big_df.columns = [
    #         "lambda","beta_tilde_1","beta_tilde_2","rho",
    #         "cost","std","delay1","delay2","delay3","delay4","hardstop1","hardstop2","hardstop3","hardstop4",
    #         "C","v"
    #     ]

    display(big_df.head())

    # save
    big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_original.csv", index=False)

In [ ]:
if USE_RAW:
    import numpy as np

    if model_name == "RS":

        lambda1, lambda3 = 0.95*1.0/3, 0.95*1.0/3  # nominal input rate
        n = 400  # scale number between original system and heavy-traffic limit
        delay_bound1, delay_bound3 = 400.0/3, 400.0/3
        b1, b3 = delay_bound1/np.sqrt(n)*lambda1, delay_bound3/np.sqrt(n)*lambda3
        b_vec = np.array([b1, b3])
        print(f"b1 = {b1.round(2)}, b3 = {b3.round(2)}")

        big_df['beta_rate_1'] = (big_df["beta_tilde_1"]/big_df["rho"]/b1).round(2)
        big_df['beta_rate_2'] = (big_df["beta_tilde_2"]/big_df["rho"]/b3).round(2)

    # elif model_name == "ECC3":
    #     lambda_ast = 1.0/2*0.95
    #     n = 400  # scale number between original system and heavy-traffic limit
    #     inv_sqrt_n = 1 / np.sqrt(n)
    #     delay_bound_v, delay_bound_h = 60, 90
    #     b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n

    #     big_df['beta_rate_1'] = (big_df["beta_tilde_1"]/big_df["rho"]/b_v).round(2)
    #     big_df['beta_rate_2'] = (big_df["beta_tilde_2"]/big_df["rho"]/b_h).round(2)
    #     big_df['eps'] = big_df['C']/np.sqrt(n)
    
    

In [ ]:
if USE_RAW:
    print("epsilon:", sorted(big_df["epsilon"].unique()))
    print("rho:", sorted(big_df["rho"].unique()))
    print("safety:", sorted(big_df["safety"].unique()))
    print("lambda:", sorted(big_df["lambda"].unique()))
    print("beta_rate_1:", sorted(big_df["beta_rate_1"].unique()))
    print("beta_rate_2:", sorted(big_df["beta_rate_2"].unique()))

In [ ]:
if USE_RAW:
    display(big_df)

In [ ]:
if USE_RAW:
    if model_name == "RS":
        selected_df = big_df[
            ["safety", "rho", "epsilon", "lambda", "beta_rate_1", "beta_rate_2",
             'cost', 'std',
             'delay1', 'delay2', 'delay1_std', 'delay2_std'
            ]
        ]
    # elif model_name == "ECC3":
    #     selected_df = big_df[
    #         ["rho", "eps", "lambda", "beta_rate_1", "beta_rate_2",
    #          "cost", "std", "delay1", "delay2", "delay3", "delay4"]
    #     ]
    selected_df = selected_df.sort_values(
        by=["safety", "rho", "epsilon", "lambda", "beta_rate_1", "beta_rate_2"]
    ).reset_index(drop=True)

In [ ]:
if USE_RAW:
    display(selected_df)

In [ ]:
if USE_RAW:
    selected_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_full.csv", index=False)

# 2. Analyze Data

## 0. read data

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os

model_name = "RS"
results_folder_name = ("cpp_results_formal_run/1_RS_proposed_policy_extra_5000p" if USE_RAW
                       else "../../2_prelimit_simulation/3_RS_saved_results/1_RS_proposed_policy_extra_5000p")
figures_folder_name = "cpp_results_formal_run/1_RS_proposed_policy_extra_5000p"

param_cols = ["safety", "rho", "epsilon", "lambda", "beta_rate_1", "beta_rate_2"]

# Outward
eps_band = "Out"

df_out = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_full.csv")

df_out = df_out.sort_values(param_cols).reset_index(drop=True)

print("\nOutward:\n")
display(df_out)

In [ ]:
epsilon_list = sorted(df_out["epsilon"].unique())
rho_list = sorted(df_out["rho"].unique())
safety_list = sorted(df_out["safety"].unique())
lambda_list = sorted(df_out["lambda"].unique())
beta_rate_1_list = sorted(df_out["beta_rate_1"].unique())
beta_rate_2_list = sorted(df_out["beta_rate_2"].unique())

print("safety:", safety_list)
print("epsilon:", epsilon_list)
print("rho:", rho_list)
print("lambda:", lambda_list)
print("beta_rate_1:", beta_rate_1_list)
print("beta_rate_2:", beta_rate_2_list)

# 1. Optimal cost and boundary check

In [ ]:
def optimal_and_boundary_check(df):
    # 1. optimal parameter row

    print("Total parameter combinations:")
    print(df.shape)
    
    df_valid = df[(df['delay1'] <= 1) & (df['delay2'] <= 1)]
    
    print("Feasible solutions (delay < 1%):")
    print(df_valid.shape)
    print("\n")
    
    best_row = df_valid.loc[df_valid['cost'].idxmin()]
    
    print("Optimal row:")
    display(best_row.to_frame().T)
    print(",".join(best_row.round(2).astype(str).values))
    print("------------------------------------------------------------------------------")
    
    # 2. inspect the grid of each parameter in the current data
    
    print("\nGrid for each parameter:\n")
    
    epsilon_list = sorted(df["epsilon"].unique())
    rho_list = sorted(df["rho"].unique())
    safety_list = sorted(df["safety"].unique())
    lambda_list = sorted(df["lambda"].unique())
    beta_rate_1_list = sorted(df["beta_rate_1"].unique())
    beta_rate_2_list = sorted(df["beta_rate_2"].unique())
    
    print("epsilon:", epsilon_list)
    print("rho:", rho_list)
    print("safety:", safety_list)
    print("lambda:", lambda_list)
    print("beta_rate_1:", beta_rate_1_list)
    print("beta_rate_2:", beta_rate_2_list)
    
    print("------------------------------------------------------------------------------")
    
    # 3. check whether each parameter is on the grid boundary
    boundary_info = []
    for p in param_cols:
        p_min = df[p].min()
        p_max = df[p].max()
        p_val = best_row[p]
        at_boundary = np.isclose(p_val, p_min) or np.isclose(p_val, p_max)
        boundary_info.append({
            "parameter": p,
            "opt_value": p_val,
            "grid_min": p_min,
            "grid_max": p_max,
            "at_boundary": at_boundary
        })
    
    boundary_df = pd.DataFrame(boundary_info)
    print("\nBoundary check:\n")
    display(boundary_df)
    
    # whether it is entirely inside the grid
    all_interior = (~boundary_df["at_boundary"]).all()
    print("Is the optimum strictly inside the grid?", all_interior)

In [ ]:
print("\nOutward:\n")
optimal_and_boundary_check(df_out)

# 2. Sensitivity to delay bound

In [ ]:
import matplotlib.pyplot as plt

def sensitivity_to_delay_threshold(df):
    records = []

    threshold_list = np.arange(0.8, 1.2, 0.01).tolist()
    
    for t in threshold_list:
        df_valid = df[(df['delay1'] <= t) & (df['delay2'] <= t)].copy()
        
        n_feasible = len(df_valid)
        
        if n_feasible == 0:
            records.append({
                'threshold': t,
                'n_feasible': 0,
                'best_cost': None,
                'rho': None,
                'epsilon': None,
                'lambda': None,
                'beta_rate_1': None,
                'beta_rate_2': None,
                'delay1': None,
                'delay2': None,
                'slack1': None,
                'slack2': None,
            })
        else:
            best_row = df_valid.loc[df_valid['cost'].idxmin()]
            records.append({
                'threshold': t,
                'n_feasible': n_feasible,
                'best_cost': best_row['cost'],
                'rho': best_row['rho'],
                'epsilon': best_row['epsilon'],
                'lambda': best_row['lambda'],
                'beta_rate_1': best_row['beta_rate_1'],
                'beta_rate_2': best_row['beta_rate_2'],
                'delay1': best_row['delay1'],
                'delay2': best_row['delay2'],
                'slack1': t - best_row['delay1'],
                'slack2': t - best_row['delay2'],
            })
    
    threshold_summary = pd.DataFrame(records)
    # display(threshold_summary)

    # import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    axes[0].plot(threshold_summary['threshold'].values, threshold_summary['n_feasible'].values, marker='o')
    axes[0].set_xlabel('delay threshold')
    axes[0].set_ylabel('number of feasible points')
    axes[0].set_title('Feasible set size vs threshold')
    axes[0].grid(True)
    
    axes[1].plot(threshold_summary['threshold'].values, threshold_summary['best_cost'].values, marker='o')
    axes[1].set_xlabel('delay threshold')
    axes[1].set_ylabel('best feasible cost')
    axes[1].set_title('Best feasible cost vs threshold')
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

In [ ]:
print("\nOutward:\n")
sensitivity_to_delay_threshold(df_out)

# 3. Best row by group

In [ ]:
def best_by_group(
    df,
    group_col,
    cost_col="cost",
    delay1_col="delay1",
    delay2_col="delay2",
    feasibility_threshold=1,
    sort_groups=True
):
    """
    For each group in group_col:
      1) find the row with minimum cost without constraints
      2) find the row with minimum cost among feasible points
         (delay1 <= threshold and delay2 <= threshold)

    Parameters
    ----------
    df : pd.DataFrame
    group_col : str
        Column to group by, e.g. "rho", "epsilon", "lambda", "safety"
    cost_col : str
        Cost column name
    delay1_col, delay2_col : str
        Delay column names
    feasibility_threshold : float
        Feasibility threshold for both delays
    sort_groups : bool
        Whether to sort output by group_col

    Returns
    -------
    best_unconstrained : pd.DataFrame
    best_constrained : pd.DataFrame
    missing_groups : list
        Groups with no feasible solution
    """

    import pandas as pd

    required_cols = [group_col, cost_col, delay1_col, delay2_col]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in df: {missing_cols}")

    # group values
    group_values = df[group_col].dropna().unique().tolist()
    if sort_groups:
        try:
            group_values = sorted(group_values)
        except Exception:
            pass

    # 1) unconstrained best
    idx_unconstrained = df.groupby(group_col)[cost_col].idxmin()
    best_unconstrained = df.loc[idx_unconstrained].copy()

    if sort_groups:
        best_unconstrained = best_unconstrained.sort_values(group_col)

    best_unconstrained = best_unconstrained.reset_index(drop=True)

    # 2) constrained best
    df_valid = df[
        (df[delay1_col] <= feasibility_threshold) &
        (df[delay2_col] <= feasibility_threshold)
    ].copy()

    best_constrained_list = []
    missing_groups = []

    for g in group_values:
        df_g_valid = df_valid[df_valid[group_col] == g]

        if len(df_g_valid) == 0:
            missing_groups.append(g)
        else:
            best_constrained_list.append(
                df_g_valid.loc[df_g_valid[cost_col].idxmin()]
            )

    if len(best_constrained_list) > 0:
        best_constrained = pd.DataFrame(best_constrained_list)
        if sort_groups:
            best_constrained = best_constrained.sort_values(group_col)
        best_constrained = best_constrained.reset_index(drop=True)
    else:
        best_constrained = pd.DataFrame(columns=df.columns)

    # print(f"Best row by {group_col} (without delay constraints):")
    # display(best_unconstrained)
    
    print(f"Best row by {group_col} (with delay constraints):")
    display(best_constrained)
    
    print(f"{group_col} with no feasible solution:")
    print(missing_groups)

    return best_unconstrained, best_constrained, missing_groups

In [ ]:
import matplotlib.pyplot as plt

def plot_best_feasible_cost(best_constrained, group_col, cost_col="cost"):
    if best_constrained.empty:
        print(f"No feasible points for {group_col}")
        return
    
    best_constrained = best_constrained.sort_values(group_col)

    plt.figure(figsize=(6, 4))
    plt.plot(best_constrained[group_col].to_numpy(), best_constrained[cost_col].to_numpy(), marker="o")
    plt.xlabel(group_col)
    plt.ylabel(cost_col)
    plt.title(f"Best feasible {cost_col} vs {group_col}")
    plt.grid(True)
    plt.show()

In [ ]:
def show_best_row_by_group(df):

    for group_col in ["rho", "epsilon", "safety", "lambda", "beta_rate_1"]:
        print("\n")
        print(f"Group by {group_col}:")
        print("\n")
        best_unconstrained, best_constrained, missing = best_by_group(
            df,
            group_col=group_col
        )
        plot_best_feasible_cost(best_constrained, group_col)

In [ ]:
print("\nOutward:\n")
show_best_row_by_group(df_out)